In [ ]:
#| default_exp core

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import contextlib, warnings
from itertools import chain
import torch

In [ ]:
#| export
def _bytes_to_mib(n: int) -> float:  # number of bytes to convert
    """Convert bytes to MiB (base-2)."""
    return n / 1024 ** 2

def _fmt_human(n: int | float) -> str:  # number to format
    """Format large number with K/M/B suffix for human-readable output."""
    if n >= 1e9: return f"{n/1e9:.2f}B"
    if n >= 1e6: return f"{n/1e6:.2f}M"
    if n >= 1e3: return f"{n/1e3:.2f}K"
    return str(int(n))


def _fmt_table(n: int | float) -> str:  # number to format
    """Format number with commas for tabular output."""
    return f"{int(n):>12,}"


def _fmt_float(v: float, width: int = 8, decimals: int = 3) -> str:  # value, width, decimal places
    """Format float with specified width and decimal places."""
    return f"{v:{width}.{decimals}f}"


def _fmt_macs(v: float) -> str:  # MACs value to format
    """Format MACs with appropriate M/K suffix."""
    if v >= 1e6: return f"{v/1e6:8.2f} M"
    if v >= 1e3: return f"{v/1e3:8.2f} K"
    return f"{int(v):>10}"


def _section(title: str, width: int = 40) -> str:  # section title, total width
    """Create a section header with box-drawing characters."""
    return f"═══ {title} " + "═" * (width - len(title))

@contextlib.contextmanager
def _device_ctx(dev: str | torch.device):  # device string or torch.device
    """Context manager that validates device availability and yields resolved device."""
    dev = torch.device(dev)
    if dev.type == "cuda" and not torch.cuda.is_available():
        warnings.warn("CUDA requested but not available – falling back to CPU")
        dev = torch.device("cpu")
    yield dev


@contextlib.contextmanager
def _on_device(model: torch.nn.Module, dev: str | torch.device):
    """Yield `model` on `dev`, then move it back where it was."""
    t = next(chain(model.parameters(), model.buffers()), None)
    orig = t.device if t is not None else dev  # no tensors: nothing to restore
    try:
        model.to(dev)
        yield model
    finally:
        model.to(orig)


def _sync(dev: torch.device) -> None:  # device to synchronize
    """Synchronize CUDA device if applicable."""
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def _default_devices() -> list[str]:
    """Return default device list: ['cpu'] + ['cuda'] if available."""
    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")
    return devices


def _is_quantized(model: torch.nn.Module) -> bool:  # model to inspect
    """True if the model contains PyTorch quantized modules.

    PyTorch's eager/FX quantized ops (fbgemm/qnnpack) are **CPU-only**. Running
    them on a CUDA tensor has no registered kernel and *segfaults* the process
    rather than raising, so callers must avoid dispatching them off-CPU.
    """
    return any("quantized" in type(m).__module__ for m in model.modules())


def _device_supported(
    model: torch.nn.Module,      # model to run
    device: str | torch.device,  # target device
) -> bool:
    """False if `model` cannot execute on `device` (quantized models are CPU-only)."""
    return not (torch.device(device).type != "cpu" and _is_quantized(model))


def _ensure_device_supported(
    model: torch.nn.Module,      # model to run
    device: str | torch.device,  # target device
) -> None:
    """Raise a *catchable* error instead of letting PyTorch segfault.

    Guards the forward-executing profilers: a quantized model dispatched on a
    non-CPU device would crash the interpreter with SIGSEGV, so we fail loudly
    (and recoverably) first.
    """
    if not _device_supported(model, device):
        raise RuntimeError(
            f"Quantized models run on CPU only; cannot benchmark on '{device}'. "
            "PyTorch quantized ops (fbgemm/qnnpack) have no CUDA kernels."
        )


def _run_on_devices(
    compute_fn,                                          # single-device compute function
    model: torch.nn.Module,                              # model to benchmark
    sample: torch.Tensor,                                # input tensor
    devices: list[str | torch.device] | None,            # devices to run on (None = default)
    nan_factory,                                         # callable(device_str) -> metrics with NaN values
    metric_name: str,                                    # for warning messages (e.g., "Speed", "Memory")
    **kwargs,
) -> dict:
    """Run compute_fn on multiple devices with unified error handling.
    
    Returns dict mapping device string to metrics (or NaN metrics on failure).
    Devices a model cannot run on (e.g. CUDA for a quantized model) are skipped
    with NaN metrics — this avoids an uncatchable SIGSEGV from dispatching
    CPU-only quantized ops on a CUDA tensor.
    """
    if devices is None:
        devices = _default_devices()

    out = {}
    for d in devices:
        d_str = str(d)
        if not _device_supported(model, d):
            warnings.warn(
                f"{metric_name} benchmark skipped on {d}: quantized models run on "
                "CPU only (PyTorch quantized ops have no CUDA kernels)."
            )
            out[d_str] = nan_factory(d_str)
            continue
        try:
            out[d_str] = compute_fn(model, sample, device=d, **kwargs)
        except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
            warnings.warn(f"{metric_name} benchmark failed on {d}: {e}")
            out[d_str] = nan_factory(d_str)
        except Exception as e:
            warnings.warn(f"Unexpected error during {metric_name} benchmark on {d}: {e}")
            out[d_str] = nan_factory(d_str)
    return out

In [ ]:
#| export
@contextlib.contextmanager
def _measuring(model: torch.nn.Module, dev: str | torch.device):
    """Yield `model` on `dev` in eval mode, then restore its device and training mode."""
    was_training = model.training
    try:
        with _on_device(model, dev) as m:
            yield m.eval()
    finally:
        model.train(was_training)

In [ ]:
#| hide
from fastcore.test import *
import torch.nn as nn

# _is_quantized: plain fp modules are not quantized
_fp = nn.Sequential(nn.Conv2d(3, 4, 3), nn.ReLU())
test_eq(_is_quantized(_fp), False)
# fp models can run anywhere; quantized models are CPU-only
test_eq(_device_supported(_fp, "cuda"), True)
test_eq(_device_supported(_fp, "cpu"), True)
_ensure_device_supported(_fp, "cuda")  # no raise for fp model

# A module whose class lives under a *.quantized.* path is detected as quantized
import torch.ao.nn.quantized as nnq
_q = nn.Sequential(nnq.Linear(4, 2))
test_eq(_is_quantized(_q), True)
test_eq(_device_supported(_q, "cpu"), True)
test_eq(_device_supported(_q, "cuda"), False)
with ExceptionExpected(RuntimeError):
    _ensure_device_supported(_q, "cuda")

# _on_device yields the model on the target device and restores the original one
_m = nn.Linear(4, 2)
with _on_device(_m, "cpu") as _inner:
    assert _inner is _m
    test_eq({p.device.type for p in _inner.parameters()}, {"cpu"})
test_eq({p.device.type for p in _m.parameters()}, {"cpu"})

# a model with neither parameters nor buffers does not raise
with _on_device(nn.ReLU(), "cpu"): pass

# _measuring yields the model in eval mode and restores the mode it came in
_tm = nn.Linear(4, 2).train()
with _measuring(_tm, "cpu") as _inner:
    assert _inner is _tm
    test_eq(_inner.training, False)
test_eq(_tm.training, True)

# ...restored even when the body raises
with ExceptionExpected(ValueError):
    with _measuring(_tm, "cpu"):
        raise ValueError("boom")
test_eq(_tm.training, True)

# an eval-mode model stays in eval mode
_m.eval()
with _measuring(_m, "cpu"): pass
test_eq(_m.training, False)

In [ ]:
#| hide
#| notest
# Needs a second device: a CUDA model measured on the CPU comes back on CUDA.
if torch.cuda.is_available():
    _cm = nn.Linear(4, 2).cuda()
    with _on_device(_cm, "cpu") as _inner:
        test_eq({p.device.type for p in _inner.parameters()}, {"cpu"})
    test_eq({p.device.type for p in _cm.parameters()}, {"cuda"})

    # ...restored even when the body raises
    with ExceptionExpected(ValueError):
        with _on_device(_cm, "cpu"):
            raise ValueError("boom")
    test_eq({p.device.type for p in _cm.parameters()}, {"cuda"})

    # _measuring restores device *and* mode
    _cm.train()
    with _measuring(_cm, "cpu") as _inner:
        test_eq({p.device.type for p in _inner.parameters()}, {"cpu"})
        test_eq(_inner.training, False)
    test_eq({p.device.type for p in _cm.parameters()}, {"cuda"})
    test_eq(_cm.training, True)

---

## See Also

- [Benchmark](../analysis/benchmark.html) — Main benchmarking API